In [10]:
# Copyright (c) 2021-2022 Cleanlab Inc.
# This file is part of cleanlab/label-errors.
#
# label-errors is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# cleanlab/label-errors is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR
# PURPOSE.  See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License

# This agreement applies to this version and all previous versions of
# cleanlab/label-errors.

"""
This tutorial provides reproducible code to find the label errors for datasets:
MNIST, CIFAR-10, CIFAR-100, ImageNet, Caltech-256, Amazon Reviews, IMDB,
20News, and AudioSet. These datasets comprise 9 of the 10 datasets on
https://labelerrors.com .

Label errors are found using the pyx (predicted probs), pred (predicted labels),
and test label files, provided in this repo: (cleanlab/label-errors)

The QuickDraw dataset is excluded because the pyx file is 33GB and might
cause trouble on some machines. To find label errors in the QuickDraw dataset,
you can download the pyx file here:
https://github.com/cleanlab/label-errors/releases/tag/quickdraw-pyx-v1

This tutorial reproduces how we find the label errors on https://labelerrors.com
(prior to human validation on mTurk).

To more closely match the label errors on labelerrors.com and in the paper,
set reproduce_labelerrors_dot_com = True
"""

import cleanlab
import numpy as np
import json
# from util import ALL_CLASSES
# To view the text data from labelerrors.com, we need:
from urllib.request import urlopen
# To view the image data from labelerrors.com, we need:
from skimage import io
from matplotlib import pyplot as plt

# Remove axes since we're plotting images, not graphs
rc = {"axes.spines.left" : False,
      "axes.spines.right" : False,
      "axes.spines.bottom" : False,
      "axes.spines.top" : False,
      "xtick.bottom" : False,
      "xtick.labelbottom" : False,
      "ytick.labelleft" : False,
      "ytick.left" : False}
plt.rcParams.update(rc)

In [11]:
datasets = [
    ('celeba', 'image'),
]

# Find label errors in each dataset

In [12]:
# By default, the code below will use the most up-to-date theory and algorithms
# of confident learning, implemented in the cleanlab package.
# We recommend this for best results.
# However, if you need to more closely match the label errors 
# to match https://labelerrors.com, set `reproduce_labelerrors_dot_com = True`.
# There may be discrepancies in counts due to improvements to cleanlab
# since the work was published.

# Set to False for best/most-recent results (this approach also runs faster)
# Set to True to match the label errors on https://labelerrors.com
reproduce_labelerrors_website = False

In [16]:
for (dataset, modality) in datasets:
    title = 'Dataset: ' + dataset.capitalize()
    print('='*len(title), title, '='*len(title), sep='\n')

    # Get the cross-validated predicted probabilities on the test set.
    pyx = np.load(f'./Male_celeba_test_set_pyx.npy', allow_pickle=True)
    # Get the cross-validated predictions (argmax of pyx) on the test set.
    pred = np.load(f'./Male_celeba_pyx_argmax_predicted_labels.npy', allow_pickle=True)
    # Get the test set labels
    labels = np.load(f'./Male_celeba_original_label.npy',
                     allow_pickle=True)
    img_paths = np.load(f"./Male_celeba_img_paths.npy")

    # Find label error indices using cleanlab in one line of code.
    # This will use the most recent version of cleanlab with best results.
    # print('Finding label errors using cleanlab for {:,} '
    #       'examples and {} classes...'.format(*pyx.shape))
    label_error_indices = cleanlab.pruning.get_noise_indices(
        s=labels,
        psx=pyx,
        # Try prune_method='both' (C+NR in the confident learning paper)
        # 'both' finds fewer errors, but can sometimes increase precision
        prune_method='prune_by_noise_rate',
        multi_label=True if dataset == 'audioset_eval_set' else False,
        sorted_index_method='self_confidence',
    )
    num_errors = len(label_error_indices)
    print('Estimated number of errors in {}:'.format(dataset), num_errors)

    # Print an example
    # Grab the first label error found with cleanlab
    err_id = label_error_indices[0]

    # Custom code to visualize each label error from each dataset
    if modality == 'image':
        url = img_paths[err_id]
        image = io.imread(url)  # read image data from a url
        plt.imshow(image, interpolation='nearest', aspect='auto', cmap='gray')
        plt.show()

    given_label = labels[err_id]
    pred_label  = pred[err_id]
    print(' * {} Given Label:'.format(dataset.capitalize()), given_label)
    print(' * We Guess (argmax prediction):', pred_label)
    print(' * Label Error Found: {}\n'.format(url))

Dataset: Celeba
Estimated number of errors in celeba: 19745


KeyboardInterrupt: 